In [19]:
PROMPT_TEMPLATE = """
You are an expert in Causal Inference. Your task is to determine the causal direction between two variables based on their context and metadata.

Dataset Context: {context}
Variable X: {var_x_desc}
Variable Y: {var_y_desc}

Based on physical laws, common sense, and scientific facts, select the most plausible causal direction:
- X -> Y (X causes Y)
- Y -> X (Y causes X)
- Independent (No direct causal relationship)

Strict Output Format (DO NOT use any markdown, do not use double asterisks ** anywhere):
Direction: [Your choice: X -> Y, Y -> X, or Independent]
Reason: [Provide a brief explanation in 1-2 sentences]
"""

In [20]:
TUEBINGEN_PAIRS = [
    {
        "pair_id": "0001",
        "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
        "var_x":"altitude",
        "var_y":"temperature (average over 1961-1990)",
        "ground_truth": "X -> Y"

    },
    {
            "pair_id": "0002",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"altitude",
            "var_y":"precipitation (yearly value averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0003",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"longitude",
            "var_y":"temperature (averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    
    },
    {
            "pair_id": "0004",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"altitude",
            "var_y":"sunshine (yearly value averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0005",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Length",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0006",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Shell weight",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0007",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Diameter",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0008",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Height",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0009",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Whole weight",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0010",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Shucked weight",
            "ground_truth": "X -> Y"
    }
]

In [ ]:
import csv
import ollama

csv_data = []
for pair in TUEBINGEN_PAIRS :
    prompt = PROMPT_TEMPLATE.format(
        context=pair["context"],
        var_x_desc=pair["var_x"],
        var_y_desc=pair["var_y"])

    response = ollama.chat(
        model='llama3.1:8B', 
        messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ],
        stream=False  # Enables real-time streaming output
    )
    output = response['message']['content']

    predicted_direction = "N/A"
    reason = "N/A"
    print(output)
    for line in output.split('\n'):
        if line.startswith("Direction:"):
            predicted_direction = line.replace("Direction:", "").strip()
        elif line.startswith("Reason:"):
            reason = line.replace("Reason:", "").strip()

    correctness = "False";
    if pair["ground_truth"] in predicted_direction or predicted_direction in pair["ground_truth"]:
        correctness = "True";
    
    csv_data.append({
        "Pair_ID": pair["pair_id"],
        "Variable_X": pair["var_x"],
        "Variable_Y": pair["var_y"],
        "Predicted_Direction": predicted_direction,
        "Ground_Truth": pair["ground_truth"],
        "Correctness": correctness,
        "Reason": reason
    })

csv_file_name = "causal_predictions.csv"
headers = ["Pair_ID", "Variable_X", "Variable_Y", "Ground_Truth", "Predicted_Direction", "Correctness", "Reason"]

with open(csv_file_name, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(csv_data)

print(f"Finish write to file: {csv_file_name}")

Direction: X -> Y
Reason: The average temperature (Y) is likely influenced by the altitude (X) of the station due to changes in atmospheric pressure and density with elevation. Additionally, physical laws such as the adiabatic lapse rate suggest that temperature decreases with increasing altitude.
Direction: X -> Y
Reason: Based on physical laws and common sense, it is well-established that altitude is a determining factor for precipitation. At higher altitudes, the atmosphere cools, leading to increased precipitation.
Direction: X -> Y
Reason: The causal direction is most plausible from longitude (X) to temperature (Y), as the physical law of climate zones and geographical variation suggests that temperature patterns are influenced by location, which is determined by longitude. This is supported by the fact that climate zones and associated temperatures generally follow longitudinal bands.
Direction: X -> Y
Reason: Altitude affects the amount of sunshine an area receives due to its im

ValueError: dict contains fields not in fieldnames: 'Correctness'